In [2]:
import os
import math
import itertools
from collections import defaultdict
from datetime import datetime


# ===============================
# Step 1. 数据加载
# ===============================
def load_data(path):
    """
    读取txt文件，返回 user -> {item: (itt, scale, shape, weight, current_itt)}
    """
    data = defaultdict(dict)
    with open(path, "r") as f:
        for line in f:
            user, item, itt, scale, shape, weight, current_itt = line.strip().split()
            data[user][item] = (
                float(itt), float(scale), float(shape), float(weight), float(current_itt)
            )
    return data


# ===============================
# Step 2. 协同过滤 (item-based Jaccard)
# ===============================
def build_item_users(train_data):
    """返回 item -> set(users)"""
    item_users = defaultdict(set)
    for user in train_data:
        for item in train_data[user]:
            item_users[item].add(user)
    return item_users


def item_similarity(item_users):
    """Jaccard 相似度"""
    sim = defaultdict(dict)
    items = list(item_users.keys())
    for i, j in itertools.combinations(items, 2):
        inter = len(item_users[i] & item_users[j])
        if inter == 0:
            continue
        union = len(item_users[i] | item_users[j])
        score = inter / union
        sim[i][j] = score
        sim[j][i] = score
    return sim


def recommend_original(user, train_data, sim_matrix, topk=50):
    """给用户做推荐（协同过滤）"""
    rank = defaultdict(float)
    interacted_items = set(train_data[user].keys())
    for item in interacted_items:
        if item not in sim_matrix: 
            continue
        for neighbor, score in sim_matrix[item].items():
            if neighbor in interacted_items:
                continue
            rank[neighbor] += score
    rec = sorted(rank.items(), key=lambda x: x[1], reverse=True)[:topk]
    return rec


# ===============================
# Step 3. 召回逻辑
# ===============================
def retrieval_list(user, train_data):
    """基于 current_itt 阈值召回"""
    rec = []
    for item, vals in train_data[user].items():
        itt, scale, shape, weight, current_itt = vals
        threshold = 0.3 if shape >= 1 else 0.7
        if current_itt > threshold:
            rec.append((item, -1))   # 分数统一设为 -1
    return rec


def final_recommend(user, train_data, sim_matrix, loss_func=0, topk=50):
    original = recommend_original(user, train_data, sim_matrix, topk)
    if loss_func == 0:
        return original
    else:
        recall = retrieval_list(user, train_data)
        # 合并并去重
        seen = set()
        final = []
        for item, score in recall + original:
            if item not in seen:
                final.append((item, score))
                seen.add(item)
        return final


# ===============================
# Step 4. Metric 类
# ===============================
class Metric(object):
    @staticmethod
    def hits(origin, res):
        hit_count = {}
        for user in origin:
            items = list(origin[user].keys())
            predicted = [item[0] for item in res[user]]
            hit_count[user] = len(set(items).intersection(set(predicted)))
        return hit_count

    @staticmethod
    def hit_ratio(origin, hits):
        total_num = sum(len(origin[user]) for user in origin)
        hit_num = sum(hits[user] for user in hits)
        return round(hit_num/total_num,5)

    @staticmethod
    def precision(hits, N):
        prec = sum([hits[user] for user in hits])
        return round(prec / (len(hits) * N),5)

    @staticmethod
    def recall(hits, origin):
        recall_list = [hits[user]/len(origin[user]) for user in hits]
        return round(sum(recall_list) / len(recall_list),5)

    @staticmethod
    def NDCG(origin,res,N):
        sum_NDCG = 0
        for user in res:
            DCG = 0
            IDCG = 0
            for n, item in enumerate(res[user]):
                if item[0] in origin[user]:
                    DCG+= 1.0/math.log(n+2,2)
            for n, item in enumerate(list(origin[user].keys())[:N]):
                IDCG+=1.0/math.log(n+2,2)
            if IDCG > 0:
                sum_NDCG += DCG / IDCG
        return round(sum_NDCG / len(res),5)


def ranking_evaluation(origin, res, N):
    measure = []
    for n in N:
        predicted = {user: res[user][:n] for user in res}
        if len(origin) != len(predicted):
            print('The Lengths of test set and predicted set do not match!')
            exit(-1)
        hits = Metric.hits(origin, predicted)
        hr = Metric.hit_ratio(origin, hits)
        prec = Metric.precision(hits, n)
        recall = Metric.recall(hits, origin)
        ndcg = Metric.NDCG(origin, predicted, n)
        measure.append(f"Top {n}\n")
        measure.append(f"Hit Ratio:{hr}\n")
        measure.append(f"Precision:{prec}\n")
        measure.append(f"Recall:{recall}\n")
        measure.append(f"NDCG:{ndcg}\n")
    return measure


# ===============================
# Step 5. 主函数
# ===============================
def run_experiment(campus_id, loss_func, topN=[3,5,10,20]):
    base_dir = "./dataset/campus_data"
    train_file = os.path.join(base_dir, f"campus_{campus_id}_train.txt")
    test_file  = os.path.join(base_dir, f"campus_{campus_id}_test.txt")

    # load
    train_data = load_data(train_file)
    test_data  = load_data(test_file)

    # build sim
    item_users = build_item_users(train_data)
    sim_matrix = item_similarity(item_users)

    # recommend
    res = {}
    for user in test_data:
        res[user] = final_recommend(user, train_data, sim_matrix, loss_func, topk=max(topN))

    # evaluate
    measures = ranking_evaluation(test_data, res, topN)

    # save
    timestamp = datetime.now().strftime("%Y-%m-%d %H-%M-%S-%f")
    save_dir = f"./results/CF/campus_{campus_id}"
    os.makedirs(save_dir, exist_ok=True)
    save_path = os.path.join(save_dir, f"CF_loss{loss_func}@{timestamp}-performance.txt")
    with open(save_path, "w") as f:
        f.writelines(measures)
    print(f"Campus {campus_id}, loss={loss_func}, results saved to {save_path}")


if __name__ == "__main__":
    campus_ids = [10, 15, 34, 102, 143]
    rep = 10
    for cid in campus_ids:
        for _ in range(rep):
            run_experiment(cid, loss_func=0)  # 方案a: CF
            run_experiment(cid, loss_func=3)  # 方案b: CF + 召回


Campus 10, loss=0, results saved to ./results/CF/campus_10/CF_loss0@2025-09-19 10-24-24-185497-performance.txt
Campus 10, loss=3, results saved to ./results/CF/campus_10/CF_loss3@2025-09-19 10-24-47-792470-performance.txt
Campus 10, loss=0, results saved to ./results/CF/campus_10/CF_loss0@2025-09-19 10-25-10-931374-performance.txt
Campus 10, loss=3, results saved to ./results/CF/campus_10/CF_loss3@2025-09-19 10-25-34-244368-performance.txt
Campus 10, loss=0, results saved to ./results/CF/campus_10/CF_loss0@2025-09-19 10-25-57-941450-performance.txt
Campus 10, loss=3, results saved to ./results/CF/campus_10/CF_loss3@2025-09-19 10-26-21-372480-performance.txt
Campus 10, loss=0, results saved to ./results/CF/campus_10/CF_loss0@2025-09-19 10-26-44-588622-performance.txt
Campus 10, loss=3, results saved to ./results/CF/campus_10/CF_loss3@2025-09-19 10-27-08-133118-performance.txt
Campus 10, loss=0, results saved to ./results/CF/campus_10/CF_loss0@2025-09-19 10-27-31-374782-performance.txt
C